© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. 

# **Catacomb Adventure**


  
    🎯
    AI National Student Olympiad Qualifier
  
  
    🧠
    Reinforcement Learning
  
  
    🏆
    100 + 15 points
  
  
    🗓️
    24 May 2025
  


**Name:** [WRITE YOUR NAME HERE]

**Contestant ID:** [WRITE YOUR CONTESTANT ID HERE]



During her travels, a brave **heroine** falls into an old, forgotten catacomb. The only way out is to **find the door that leads to the surface**. The catacomb, however, is full of dangers - deep pits, traps and roaming **zombies** that patrol the dark corridors following various patterns.

Help the heroine escape the catacomb using **reinforcement learning** (RL) and **deep reinforcement learning** (Deep RL) techniques. The algorithm's task is to learn the optimal route from the entrance to the exit while avoiding the dangers, **gathering experience through rewards** during learning.

## **The Environment (Gymnasium)**

**Gymnasium** is a **standardized API** for reinforcement learning (RL), designed to be easy to use, flexible and broadly applicable for modelling various RL problems.

The goal of **Gymnasium** is to offer a simple, "*pythonic*" **interface** that makes it easy to model and test different learning algorithms. A major advantage of the library is that it **standardizes communication with RL environments**, so users can easily focus on developing learning algorithms **instead of having to write a separate interface for every environment**.

For this task we **created a custom Gymnasium environment** that fully follows the standardized API described above. Thanks to this, the catacomb simulation became easy to handle and **readily integrable with any reinforcement learning algorithm**.

## **Required Imports**

In [ ]:
!pip install -qq creepy-catacombs-s1==0.1.4

from tqdm import tqdm
import numpy as np

import matplotlib.pyplot as plt
import creepy_catacombs_s1
import gymnasium as gym

### **Trying Out the Environment**

With the two lines below we **create the catacomb environment** and then reset it to its initial state using the `reset` function. The `render_mode='rgb_array'` parameter ensures that the environment can also be displayed visually. `reset` not only restarts the environment, **but also places the heroine back at the catacomb entrance**, into the starting position. Setting the `seed` is important because every different seed generates a different catacomb structure.

`reset` returns two values: `obs` (**observation**) contains the agent's first observation, for example its own position, while the dictionary called `info` provides supplementary information that can be useful during the learning process.

For offline use the `render_mode='human'` setting works perfectly well, since it allows the environment to be visualized in a separate window using Pygame. It is important to note, however, that this solution requires a graphical interface (X server), so it **does not work on Google Colab**, where there is no native support for windowed display. On Colab it is recommended to use other rendering modes, such as `rgb_array` (the `plot_env` function below can help with this).


In [ ]:
def plot_env(rgb_array: np.ndarray):
    """
    Plots the provided RGB array as an image.

    Args:
        rgb_array (np.ndarray): A NumPy array representing an RGB image.
                                 The array must have shape (height, width, 3),
                                 where the last dimension represents the RGB color channels.
    """
    if not isinstance(rgb_array, np.ndarray):
        raise TypeError("rgb_array must be a NumPy array.")

    if rgb_array.ndim != 3 or rgb_array.shape[2] != 3:
        raise ValueError("rgb_array must have shape (height, width, 3).")

    plt.figure(figsize=(8, 8))
    plt.imshow(rgb_array)
    plt.axis('off')
    plt.show()

In [ ]:
env = gym.make("CreepyCatacombs-v0", render_mode="rgb_array")
obs, info = env.reset(seed=42)
env_rgb = env.render()

In [ ]:
plot_env(env_rgb)

As you can see, the default environment **currently contains no zombies** - but this can easily be changed. The environment can handle two further arguments: the `n_zombies` parameter determines how many zombies appear in the catacomb, while the `zombie_movement` parameter controls what movement pattern these zombies follow (`random`, `towards_player`).

In [ ]:
env = gym.make("CreepyCatacombs-v0", render_mode="rgb_array", n_zombies = 3, zombie_movement = "random")
obs, info = env.reset(seed=42)
env_rgb = env.render()

In [ ]:
plot_env(env_rgb)

## **Observation, Info**

Now let's look at exactly what `observation` and `info` contain. These are the data you get back at every single interaction with the environment, so they play a key role in the agent's learning and decision making. The `observation` describes the agent's current point of view - for example its position or its immediate surroundings -, while `info` may contain supplementary information about the map, the location of the goal or other events that can be useful during analysis and visualization.

In [ ]:
print("Observation space:", obs) # Which cell the heroine is in. The origin is in the top-left corner -> 7x7 + 1 - 1 (due to indexing) = 49
print("Agent position: (%d, %d)" % (int(info['agent_pos'][0]), int(info['agent_pos'][1])))
print("Goal position: (%d, %d)" % (int(info['goal_pos'][0]), int(info['goal_pos'][1])))

zombie_positions = [(int(z[0]), int(z[1])) for z in info['zombies']]
print("Zombie positions:", zombie_positions)
print("Full environment (0 - free cell, -1 - wall, -2 - pit, 1 - starting position, 2 - goal position, 3 - zombie):\n", info['grid'])
print("Environment size: %dx%d" % (int(info["grid_shape"][0]), int(info["grid_shape"][1])))

plothole_positions = [tuple(int(p) for p in plothole) for plothole in info['plotholes']]
print("Pit positions", plothole_positions)

## **Stepping the Environment**

For the heroine to take steps in the catacomb (for example up, down, right or left), we have to call the environment's `.step()` function. This function expects an integer as input that encodes the desired direction. The four possible steps are encoded as follows:

- `0` - step **up**
- `1` - step **right**
- `2` - step **down**
- `3` - step **left**

After every `.step()` call the environment returns the new observation (`observation`), the reward received (`reward`), a boolean value (`done`) indicating whether the episode has ended, and the `info` dictionary with the supplementary data. **An episode** in reinforcement learning means a complete run through the environment, which starts from the initial state and **lasts until the agent reaches the goal or a terminal state** (e.g. it falls into a pit, or a zombie attacks it). The `truncated` variable indicates when the episode ended not naturally (e.g. by reaching the goal), but **because the maximum allowed number of steps was exceeded**. This means the agent wandered around the environment for too long without reaching the exit or making a mistake, so the system automatically closed the episode.

In [ ]:
FEL = 0
JOBBRA = 1
LE = 2
BALRA = 3

obs, reward, done, truncated, info = env.step(1)
env_rgb = env.render()
plot_env(env_rgb)

In [ ]:
print("Reward received for the step: ", reward)
print("Has the episode ended? ", done)

## **Reward System**

In the catacombs environment a **simple reward system** supports the agent's learning. The goal is for the heroine to reach the exit quickly, efficiently and safely, while avoiding the dangers. The table below summarizes what reward or penalty is given in each situation:

| Event                                        | Reward value   | Note                                        |
|---------------------------------------------|----------------|---------------------------------------------|
| Every step                                  | -1             | Encourages a quick escape                   |
| Illegal step into a wall                    | -5             | Penalty for bumping into a wall             |
| Falling into a pit or colliding with a zombie | -100           | Episode ends (`done = True`)                |
| Reaching the goal (getting to the exit)     | +0             | Episode ends (`done = True`)                |


# **Task 1 (5 points)**

Your first task is to create **a predefined sequence of steps** (a hardcoded solution, **without zombies**) that successfully leads the heroine to the catacomb exit on the deterministically generated map with `seed=2025`.

During the implementation, pay attention to the following:

- At the end of the simulation, display:
  - the heroine's **final position** (when she steps onto the exit),
  - the **reward value** collected during the episode (`total_reward`).

In [ ]:
env = gym.make("CreepyCatacombs-v0", render_mode="rgb_array")
env.reset(seed=2025)
cum_reward = 0

# --------- YOUR OWN SOLUTION GOES HERE --------- (~5-7 lines)

# ----------- END OF YOUR SOLUTION -------------

env_rgb = env.render()
plot_env(env_rgb)
print("Total reward:", cum_reward)
env.close()

# **Task 2 (20 points)**

Create a **simple pathfinding algorithm** (e.g. backtracking or BFS) that finds a valid path for the heroine from the starting point to the exit, avoiding walls and obstacles.

Restart the environment with `seed=2025`. Step the heroine along the path you found, and at the end of the simulation print:
- the final state of the environment (render)
- the total reward

In [ ]:
env = gym.make("CreepyCatacombs-v0", render_mode="rgb_array")

def search_door(env: gym.Env, seed: int) -> int:
    obs, info = env.reset(seed=seed)
    agent_pos = tuple(info['agent_pos'])
    goal_pos = tuple(info['goal_pos'])
    grid = info['grid']
    total_reward = 0
    # --------- YOUR OWN SOLUTION GOES HERE --------- (~20-30 lines)

    # ----------- END OF YOUR SOLUTION -------------
    return total_reward

cum_reward = search_door(env, seed=2025)
env_rgb = env.render()
plot_env(env_rgb)
print("Total reward:", cum_reward)
env.close()

# **Task 3 (10 points)**

Test the heuristic solution you created earlier in environments where 1, 5, and 9 zombies are present. In each case:

- Set `zombie_movement` to random.
- Set the appropriate `n_zombies` value when creating the environment.
- Run **10 episodes** with each given setting.
- At the end of every episode, record the **total reward** achieved.
- For each run, restart the environment with a unique but deterministic seed: use the seed `2025` for the first episode, then `2025 + episode_index` for every subsequent episode.


Plot the results on a single, shared plot, where the x-axis is the episode index and the y-axis is the reward for that episode. All three cases (1, 5, 9) should appear on the figure, distinguished by different colors or markers.

In [ ]:
results = {}

# --------- YOUR OWN SOLUTION GOES HERE --------- (~15 lines)

# ----------- END OF YOUR SOLUTION -------------

env.close()

# **Task 4 (15 points)**

Extend **the pathfinding algorithm you created in the previous task** so that the heroine tries to avoid not only the walls and holes, **but the zombies as well when planning her route**.

The goal is still to reach the exit safely.

In [ ]:
env = gym.make("CreepyCatacombs-v0", render_mode="rgb_array")

def search_door_avoid_zombies(env: gym.Env, seed: int) -> int:
    obs, info = env.reset(seed=seed)
    agent_pos = tuple(info['agent_pos'])
    goal_pos = tuple(info['goal_pos'])
    grid = info['grid']
    total_reward = 0
    # --------- YOUR OWN SOLUTION GOES HERE --------- (~20-30 lines)

    # ----------- END OF YOUR SOLUTION -------------
    return total_reward

results = {}

# --------- YOUR OWN SOLUTION GOES HERE --------- (~15 lines)

# ----------- END OF YOUR SOLUTION -------------
env.close()

# **Task 5 (20 points)**

**Monte Carlo Control** is a model-free reinforcement learning algorithm whose goal is to find the optimal policy in an environment where the dynamics of the environment (that is, what the next state and reward will be when a given action is taken from a given state) are unknown. The name *Monte Carlo* refers to the fact that the algorithm relies on random sampling (episodes) for learning.

### Basic concepts

* **State ($s$):** The current situation of the environment at a given moment.
* **Action ($a$):** A step the agent can take in the environment.
* **Reward ($r$):** The feedback given by the environment in response to an action of the agent.
* **Episode:** A complete sequence of states, actions, and rewards that runs from an initial state to a terminal state.
* **Return ($G$):** The sum of all rewards received during an episode, taking the discount factor ($\gamma$) into account. The return starting from time step $t$:

    $$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots + \gamma^{T-t-1} R_T$$
    where $T$ is the end of the episode.
* **Q-value (Action-Value - $Q(s, a)$):** The expected return obtained by taking action $a$ in state $s$ and following the optimal policy thereafter. Monte Carlo Control **estimates the Q-values directly.**
* **Policy ($\pi(a|s)$):** A rule that specifies with what probability the agent selects a given action in a given state. The goal of Monte Carlo Control is to find the optimal policy ($\pi^*$) that maximizes the expected return.
* **Epsilon-greedy policy ($\epsilon$-greedy policy):** A policy in which the agent most of the time selects the action that is best according to the current estimates (**exploitation**), but with a small probability ($\epsilon$) selects an action at random (**exploration**). This ensures that the agent does not get stuck with a suboptimal policy.

### **Steps of the Monte Carlo Control Algorithm**

The goal of Monte Carlo Control is **to iteratively improve the Q-values based on the experience gathered from episodes until the optimal Q-values are reached**, from which the optimal policy can be derived. The typical steps of the algorithm are the following:

1.  **Initialization:**
    * Initialize the Q-value table $Q(s, a)$ for every possible $(s, a)$ pair (usually to zero or to random values).
    * Initialize an empty dictionary for storing the returns, for example `returns[(s, a)] = []`. This dictionary stores, for every (state, action) pair, the returns obtained in the episodes so far.
    * Specify the discount factor ($\gamma$) and the initial value of epsilon ($\epsilon$).

2.  **Generating episodes:**
    * In every iteration (episode) the agent starts from the initial state and interacts with the environment according to its current policy (for example the $\epsilon$-greedy policy based on the current Q-values).
    * The episode continues until a terminal state is reached.
    * During the episode we record the sequence of states, actions, and rewards: $s_0, a_0, r_1, s_1, a_1, r_2, \dots, s_{T-1}, a_{T-1}, r_T, s_T$.

3.  **Computing the returns:**
    * After the episode has finished, we compute the return ($G_t$) for every time step $t=0, 1, \dots, T-1$. The return is the discounted sum of all future rewards starting from that time step.

4.  **Updating the Q-values:**
    * For every state-action pair $(s_t, a_t)$ that occurred in the episode, we append the computed return ($G_t$) to the list `returns[(s_t, a_t)]`.
    * We update the value of $Q(s_t, a_t)$ with the average of the returns collected so far for that (state, action) pair:
        $$Q(s_t, a_t) \leftarrow \text{mean}(returns[(s_t, a_t)])$$
    * **First-Visit MC vs. Every-Visit MC:** The step above can be "first-visit", meaning that within an episode we update the Q-value for a given $(s, a)$ pair only at its first occurrence, or "every-visit", meaning that we update it every single time the pair occurs in the episode. The code provided uses the **every-visit** approach.

5.  **Policy improvement:**
    * At the end of the episode (or at certain intervals) we improve the policy based on the updated Q-values. For example, in the case of the $\epsilon$-greedy policy, in every state we select the action with the highest Q-value, except with a small probability ($\epsilon$), when we select an action at random.
    * The value of $\epsilon$ is often decreased as the episodes progress (epsilon decay), so that the algorithm increasingly focuses on the best actions it has already learned (exploitation).

6.  **Repetition:**
    * Repeat steps 2-5 for the specified number of episodes or until the Q-values converge.

In [ ]:
# Helper function for plotting
def moving_average(data, window_size=50):
    if len(data) < window_size:
        return np.convolve(data, np.ones(window_size) / window_size, mode='valid')
    return np.convolve(data, np.ones(window_size) / window_size, mode='valid')

def monte_carlo_control_estimation(env, episodes=2500, gamma=0.99, epsilon=1.0, epsilon_decay=0.999, min_epsilon=0.01):
    """
    Implementation of the Monte Carlo Control estimation algorithm.

    Args:
        env (gym.Env): The environment the agent interacts with.
        episodes (int): The number of episodes the algorithm runs for.
        gamma (float): The discount factor (between 0 and 1). It determines the present value of future rewards.
        epsilon (float): The initial epsilon value for the epsilon-greedy policy.
        epsilon_decay (float): The epsilon decay factor. After every episode the epsilon value is multiplied by this.
        min_epsilon (float): The minimum epsilon value that epsilon can decay to.

    Returns:
        tuple: Contains the smoothed rewards, the optimal Q-value table and the optimal V-value table.
    """
    # Querying the sizes of the environment's state space and action space
    # 8x7
    n_states = env.observation_space.n
    # 4
    n_actions = env.action_space.n

    # We initialize the Q-value table (state-action values). Every value is 0 initially.
    Q = np.zeros((n_states, n_actions))
    # We initialize the 'returns' dictionary, which stores the returns collected so far for each (state, action) pair.
    returns = {}
    # We initialize the list of cumulative rewards, which stores the total reward achieved in each episode.
    cum_rewards = []

    # We iterate over the specified number of episodes
    for ep in tqdm(range(episodes)):
        # At the beginning of the episode we reset the environment to the initial state.
        state, info = env.reset(seed = 2025)
        # List for storing the steps of the current episode. Each step is a (state, action, reward) triple.
        episode = []
        # Indicates whether the episode has finished (we have reached a terminal state).
        done = False
        # Indicates whether the episode was interrupted for some reason (e.g. time limit).
        truncated = False
        # Variable for tracking the total reward of the current episode.
        total_reward = 0

        # --------- YOUR OWN SOLUTION GOES HERE --------- (~15-20 lines)

        # ----------- END OF YOUR SOLUTION -------------

        # Decreasing the value of epsilon in order to reduce the amount of exploration as the episodes progress.
        epsilon = max(min_epsilon, epsilon * epsilon_decay)

    # The optimal V-value table (state values) equals the maximum of the rows of the Q-value table.
    V = np.max(Q, axis=1)
    # We determine the moving average window size for smoothing the rewards for plotting.
    plot_window_size = min(50, episodes // 5 if episodes >= 5 else 1)
    # We compute the smoothed rewards using the moving average.
    smoothed_rewards = moving_average(cum_rewards, window_size=50)
    # We create an x-axis for plotting the smoothed rewards.
    smoothed_x = np.arange(plot_window_size - 1, len(cum_rewards))

    # Plotting how the rewards evolve over the episodes.
    plt.figure(figsize=(12, 6))
    plt.plot(smoothed_x, smoothed_rewards, label=f"Averaged Rewards (Window size={plot_window_size})", color='blue')
    plt.xlabel("Episode")
    plt.ylabel("Total reward")
    plt.title("Monte Carlo Control: Total Reward per Episode")
    plt.legend()
    plt.grid(True)
    plt.show()

    # We return the smoothed rewards, the optimal Q-value table and the optimal V-value table.
    return smoothed_rewards, Q, V

In [ ]:
env = gym.make("CreepyCatacombs-v0", render_mode="rgb_array")
smoothed_rewards_MC, Q_table_MC, V_table_MC = monte_carlo_control_estimation(env,
                                                  episodes=2500,
                                                  gamma=0.99,
                                                  epsilon=1.0,
                                                  epsilon_decay=0.999,
                                                  min_epsilon=0.01)

# **Task 6 (5 points)**

The `env` environment provides you with three visualization functions that let you display the values and policies learned by the algorithm. These functions return an RGB image (array), which you can easily display with the previously defined `plot_env()` function.

**The three available visualizations:**

- `env.unwrapped.render_values(V)`  
  It expects the values belonging to the states (`state: value`) in the form of a dictionary.  
  This function returns a visualization of the value function \(V(s)\).

- `env.unwrapped.render_q_values(Q)`  
  It expects state-action pairs as a dictionary (`(state, action): Q-value`) and displays the Q-values.  
  This can be used to visualize the \(Q(s, a)\) value function.

- `env.unwrapped.render_optimal_path(policy)`  
  It expects the policy containing the best actions (`np.argmax(Q_table, axis=1)`) and displays the optimal path in the policy.

Every function returns an RGB image, so it can be easily rendered with the `plot_env` function.

In [ ]:
state, info = env.reset(seed=2025)

# --------- YOUR OWN SOLUTION GOES HERE --------- (~10 lines)

# ----------- END OF YOUR SOLUTION -------------

env.close()

# **Task 7 (20 points)**

**SARSA** (State-Action-Reward-State-Action) is an on-policy temporal difference (**Temporal Difference - TD**) learning algorithm used in reinforcement learning to find the optimal policy. The term *on-policy* simply means that the algorithm learns with the very policy it is currently using to select actions.

### Basic concepts

* **State ($s$):** The current situation of the environment at a given moment.
* **Action ($a$):** A step the agent can take in the environment.
* **Reward ($r$):** The feedback given by the environment in response to an action of the agent.
* **Q-value (Action-Value - $Q(s, a)$):** The expected return obtained if we take action $a$ in state $s$ and then follow the current policy. SARSA estimates the Q-values directly.
* **Policy ($\pi(a|s)$):** A rule specifying with what probability the agent chooses a given action in a given state. **SARSA learns the Q-values of a given policy.**
* **Learning rate ($\alpha$):** A parameter that determines how strongly new experiences influence the previous Q-values. A higher learning rate means that new experiences carry more weight.
* **Discount factor ($\gamma$):** A parameter (between 0 and 1) that determines the present value of future rewards. A lower discount factor favours immediate rewards, while a higher value also takes long-term rewards into account.
* **Epsilon-greedy policy ($\epsilon$-greedy policy):** A policy in which the agent most of the time chooses the action that is best according to the current estimates (**exploitation**), but with a small probability ($\epsilon$) it chooses an action at random (**exploration**).

### **Steps of the SARSA Algorithm**

During an episode, the SARSA algorithm updates the Q-values step by step as follows:

1.  **Initialisation:**
    * We initialise the Q-value table $Q(s, a)$ for every possible $(s, a)$ pair (usually to zero or to random values).
    * We specify the learning rate ($\alpha$), the discount factor ($\gamma$) and the initial value of epsilon ($\epsilon$).

2.  **Running the Episodes:**
    * In every episode we start from the initial state ($s_t$).
    * Using the current policy (e.g. $\epsilon$-greedy based on the current Q-values) we select an action ($a_t$) in state $s_t$.
    * We execute action $a_t$ and observe the next state ($s_{t+1}$) and the reward received ($r_{t+1}$).
    * In the next state ($s_{t+1}$) we again select an action ($a_{t+1}$) using the same policy.
    * We update the Q-value based on the transition just experienced, using the following formula:
        $$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$$
    * The current state and action become the state and action of the next step ($s_t \leftarrow s_{t+1}$, $a_t \leftarrow a_{t+1}$).
    * We repeat this process until we reach a terminal state.

3.  **Decaying Epsilon:**
    * As the episodes progress, we can decrease the value of epsilon (epsilon decay), so that the agent increasingly concentrates on the best actions it has already learned (exploitation) instead of acting randomly (exploration).

4.  **Repetition:**
    * We repeat steps 2 and 3 for the given number of episodes or **until the Q-values converge.**

In [ ]:
# Helper function for the SARSA algorithm
def epsilon_greedy_policy(Q, state, epsilon, n_actions):
    """
    Implements the epsilon-greedy policy for action selection.

    Args:
        Q (numpy.ndarray): The Q-value table (state-action values).
        state (int): The current state.
        epsilon (float): The probability of choosing a random action (exploration).
        n_actions (int): The number of available actions.

    Returns:
        int: The index of the selected action.
    """
    if np.random.rand() < epsilon:
        # If a random number is smaller than epsilon, we choose an action at random (exploration).
        return np.random.choice(n_actions)
    else:
        # Otherwise we choose the action with the highest Q-value in the current state (exploitation).
        q_values = Q[state, :]
        max_q = np.max(q_values)
        best_actions = np.where(q_values == max_q)[0]
        # If there are several best actions, we choose one of them at random.
        return np.random.choice(best_actions)

def sarsa_modified(env, episodes=1000, alpha=0.1, gamma=0.99,
                    epsilon=1.0, epsilon_decay=0.999, min_epsilon=0.01):
    """
    Implementation of the SARSA (State-Action-Reward-State-Action) algorithm.

    Args:
        env (gym.Env): The environment the agent interacts with.
        episodes (int): The number of episodes the algorithm runs for.
        alpha (float): The learning rate (between 0 and 1). Determines how much we update the Q-values based on new experiences.
        gamma (float): The discount factor (between 0 and 1). Determines the present value of future rewards.
        epsilon (float): The initial epsilon value for the epsilon-greedy policy.
        epsilon_decay (float): The epsilon decay factor. After each episode the epsilon value is multiplied by it.
        min_epsilon (float): The minimum epsilon value down to which epsilon may decrease.

    Returns:
        tuple: Contains the smoothed rewards, the optimal Q-value table and the optimal V-value table.
    """
    # Querying the sizes of the environment's state space and action space
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    # We initialise the Q-value table (state-action values). Every value is 0 initially.
    Q = np.zeros((n_states, n_actions))

    # We initialise the list of cumulative rewards, which stores the total reward achieved in each episode.
    cum_rewards = []
    # We set the current epsilon value to the initial value.
    current_epsilon = epsilon

    # We iterate through the given number of episodes
    for ep in tqdm(range(episodes)):
        # At the beginning of the episode we reset the environment to the initial state.
        state, info = env.reset(seed = 2025)
        # Variable for tracking the total reward of the current episode.
        total_reward = 0
        # We select an action based on the epsilon-greedy policy in the initial state.
        action = epsilon_greedy_policy(Q, state, current_epsilon, n_actions)
        # Indicates whether the episode has finished (we have reached the terminal state).
        done = False
        # Indicates whether the episode was truncated for some reason (e.g. time limit).
        truncated = False
        # --------- YOUR OWN SOLUTION GOES HERE --------- (~10 lines)

        # ----------- END OF YOUR SOLUTION -------------
        cum_rewards.append(total_reward)

        # Decaying the epsilon value in order to reduce the amount of exploration as the episodes progress.
        current_epsilon = max(min_epsilon, current_epsilon * epsilon_decay)

    # The optimal V-value table (state values) equals the maximum of the rows of the Q-value table.
    V = np.max(Q, axis=1)

    # We determine the moving-average window size for smoothing the rewards for plotting.
    plot_window_size = min(100, episodes // 5 if episodes >= 5 else 1)
    # We compute the smoothed rewards using the moving average.
    smoothed_rewards = moving_average(cum_rewards, window_size=plot_window_size)
    # We create an x-axis for plotting the smoothed rewards.
    smoothed_x = np.arange(plot_window_size - 1, len(cum_rewards))

    # Plotting the evolution of the rewards over the episodes.
    plt.figure(figsize=(12, 6))
    plt.plot(smoothed_x, smoothed_rewards, label=f"Averaged Rewards (Window Size={plot_window_size})", color='red')
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.title(f"SARSA: Total Reward per Episode (α={alpha}, γ={gamma})")
    plt.legend()
    plt.grid(True)
    plt.show()

    # We return the smoothed rewards, the optimal Q-value table and the optimal V-value table.
    return smoothed_rewards, Q, V

In [ ]:
env = gym.make("CreepyCatacombs-v0", render_mode="rgb_array")
smoothed_rewards_sarsa, Q_table_sarsa, V_table_sarsa = sarsa_modified(env,
                                              episodes=2500,
                                              alpha=0.1,
                                              gamma=0.99,
                                              epsilon=1.0,
                                              epsilon_decay=0.999,
                                              min_epsilon=0.01)

# **Task 8 (5 points)**

Repeat the modifications you made in **Task 6**, this time applying them to the **SARSA algorithm**!


In [ ]:
state, info = env.reset(seed=2025)
# --------- YOUR OWN SOLUTION GOES HERE --------- (~10 lines)

# ----------- END OF YOUR SOLUTION -------------
env.close()

# **Task 9 (+15 points, extra)**

Create a joint plot comparing the **rewards smoothed with a moving average** for the Monte Carlo (MC) and the SARSA algorithms. Use the same environment and the same learning parameters for both methods.

Then run the MC algorithm in an environment containing **2 zombies** with a **random movement pattern** (`zombie_movement="random"`), and examine how learning changes compared with the original environment.

During your investigation, take into account the modifications required in **Tasks 6 and 8**, and describe your observations about the differences.

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~8 lines)
# Plot
# ----------- END OF YOUR SOLUTION -------------

env = gym.make("CreepyCatacombs-v0", render_mode="rgb_array", n_zombies = 2, zombie_movement = "random")
smoothed_rewards_MC_zombie, Q_table_MC_zombie, V_table_MC_zombie = monte_carlo_control_estimation(env,
                                                  episodes=2500,
                                                  gamma=0.99,
                                                  epsilon=1.0,
                                                  epsilon_decay=0.999,
                                                  min_epsilon=0.01)
state, info = env.reset(seed=2025)
# --------- YOUR OWN SOLUTION GOES HERE --------- (~10 lines)

# ----------- END OF YOUR SOLUTION -------------
env.close()

---

## 🎉 Congratulations!

You have reached the end of the problem set - you did an excellent job!  

---